In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

import math

class MultiHeadAttention:
    def __init__(self, n_heads, embd_size):
        self.n_heads = n_heads
        self.embd_size = embd_size

        assert self.embd_size % self.n_heads == 0
        self.head_embd_size = self.embd_size // self.n_heads

        self.QKV = nn.Linear(self.embd_size, self.n_heads * 3*self.head_embd_size)
        self.W_O = nn.Linear(self.embd_size, self.embd_size)

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        T, embd_size = x.shape

        qkv = self.QKV(x).reshape(T, self.n_heads, -1).transpose(1, 0, 2) # (n_heads, T, 3*head_embd_size)
        Q, K, V = qkv.split(3, dim = -1) # (n_heads, T, head_embd_size)

        # (n_heads, T, T)
        attn_scores = torch.softmax((Q @ K.tranpose(0, 2, 1)) / math.sqrt(self.head_embd_size), dim = -1)
        z = attn_scores @ V # (n_heads, T, head_embd_size)
        z = z.transpose(1, 0, 2).reshape(T, -1) # (T, n_heads * head_embd_size) = (T, embd_size)
        z = z @ self.W_O # (T, embd_size)

        return z



